In [ ]:
import os, sys, warnings, requests, pandas as pd
from bs4 import BeautifulSoup
warnings.filterwarnings("ignore")
sys.path.append('/dfs/user/hanchenw/BioAgentOS')

from bioagentos.agent.codeact import codeact_agent
from bioagentos.utils import load_pkl

### CodeAgent

In [ ]:
tool_index = "all_250324"
llm_name =  "claude-3-7-sonnet-20250219"
tools = load_pkl(f'/dfs/user/hanchenw/BioAgentOS/bioagentos/tool/{tool_index}.pkl')

In [ ]:
coding_tool = {}
coding_tool['bioagentos.tool.support_tools'] = tools['bioagentos.tool.support_tools']

In [ ]:
# codeact
agent = codeact_agent(coding_tool, llm_name)
agent.configure()

### Load GEO Metadata

In [ ]:
def get_geo_metadata(gse_id: str):
    """
    Fetch metadata for a GEO dataset given its GSE ID.

    Args:
        gse_id (str): GEO Series accession number (e.g. 'GSE122080')

    Returns:
        dict: Dictionary containing:
            - title: Study title
            - summary: Study description/summary
            - overall_design: Study design information

    Raises:
        ValueError: If GSE ID is invalid or data cannot be retrieved
        ConnectionError: If there are network issues
    """

    # Validate GSE ID format
    if not gse_id.startswith("GSE") or not gse_id[3:].isdigit():
        raise ValueError(
            "Invalid GSE ID format. Must start with 'GSE' followed by numbers"
        )

    # Use GEO's direct API
    url = f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gse_id}&targ=self&form=xml&view=quick"

    try:
        # Fetch the record
        response = requests.get(url)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "xml")

        # Find the Series element
        series = soup.find("Series")
        if not series:
            raise ValueError(f"No metadata found for GSE ID: {gse_id}")

        # Extract metadata with error handling
        try:
            title = series.find("Title").text.strip()
        except (AttributeError, TypeError):
            title = ""

        try:
            summary = series.find("Summary").text.strip()
        except (AttributeError, TypeError):
            summary = ""

        try:
            design = series.find("Overall-Design").text.strip()
        except (AttributeError, TypeError):
            design = ""

        # Validate that we got at least some data
        if not any([title, summary, design]):
            raise ValueError(f"No metadata content found for GSE ID: {gse_id}")

        return {
            "title": title,
            "summary": summary,
            "overall_design": design,
        }

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to fetch data from GEO: {str(e)}")
    except Exception as e:
        raise ValueError(f"Failed to parse GEO response: {str(e)}")


# Example usage:
metadata = get_geo_metadata("GSE263223")
print("Title:", metadata["title"])
print("\nSummary:", metadata["summary"])
print("\nStudy Design:", metadata["overall_design"])

In [ ]:
metadata

In [ ]:
study_id = "GSE66380"
study_design = get_geo_metadata(study_id)
study_design

### Prompt 1 (CSV + Report)
    $0.8 - 0.9 each case

In [ ]:
study_id = "GSE122080" # "GSE263223"
study_design = get_geo_metadata(study_id)

base_dir = "/dfs/user/hanchenw/Agent_PG/clockbase_agent/"
data_dir = f"{base_dir}/data"

task_intention = f'''
Analyze the aging effects of conditions from study {study_id}.

Workflow:

1. Data Acquisition and Preparation
    - Load dataset overview from: `{data_dir}/dataset_level/gse_rna_mm.csv` (Filter by "accession" column to locate {study_id})
    - Extract metadata from: `{data_dir}/GEO_mm_rna/processed_metadata_preharmonize/{study_id}.csv`
    - Import pre-calculated aging clock data from: `{data_dir}/clock_output/mouse_rna_age_short.csv`
    - Create output directory if needed: `/dfs/user/hanchenw/Agent_PG/clockbase_agent/result/{study_id}/`


2. Study Design Information:
   {study_design}

3. Clock Selection and Analysis:
    - **IMPORTANT: DO NOT compare young and old populations directly**
    - **Target Aging Clocks:**
      * `Hazard_Rodents_All_All_scaleddiff_EN_tAge`
      * `Hazard_Mouse_All_All_scaleddiff_EN_tAge`
      
    - **For Each Condition:**
      * Identify appropriate control group(s) based on metadata
      * Determine relevant covariates (age, sex, tissue type, batch effects)
      * Focus on comparing treatment vs. control within the same age group
      * Perform statistical analysis:
        - Use appropriate tests based on data structure (t-test, ANOVA, or linear models)
        - Report raw p-values (no adjustment required)
        - Calculate standardized effect sizes (Cohen's d preferred)

4. Results Interpretation
    - Evaluate biological significance of findings
    - Compare consistency between different clock measures
    - Assess potential mechanisms and novelty of the interventions
    - Document limitations and confounding factors
    - Ensure all interpretations avoid direct young vs. old comparisons

5. Deliverables:
   - Comprehensive analysis report with methods, results, and interpretation
   - Standardized results table with these columns:
     * gse_id: The dataset identifier ({study_id})
     * comparison_name: Descriptive name of the comparison (e.g., "Drug X vs Vehicle")
     * condition_name: Concise intervention name (e.g., "Rapamycin")
     * condition_category: Categorization (one of: Genetic, Drug, Diet, Environment, Disease, Other)
     * covariate: Variables controlled for in analysis
     * clock: Specific aging clock used
     * statistical_test: Method used (e.g., "ANOVA", "Linear Mixed Model")
     * p_value: Raw p-value (do NOT use adjusted p-values)
     * effect_size: Magnitude of effect (Cohen's d or similar)
     * direction: Age effect direction (1=accelerated aging, -1=decelerated aging)
     * trust_score: Evaluate experimental follow-up value (0-10):
     
         Trust Score Criteria
            - **0-3: Low value**
              - Likely technical artifact
              - Limited biological plausibility
              - Insufficient replication
            
            - **4-6: Moderate value**
              - Some biological plausibility
              - Limited supporting literature
              - Adequate experimental design
            
            - **7-8: High value**
              - Strong biological plausibility
              - Supported by published mechanisms
              - Robust experimental design
            
            - **9-10: Exceptional value**
              - Novel with transformative potential
              - Strong literature support for mechanism
              - Exceptional rigor in design and analysis

Implementation Requirements:
    - Use Python with pandas, scipy, and statsmodels
    - Implement modular code structure with separate functions for:
      * Data loading and preprocessing
      * Statistical analysis
      * Results formatting
    - Include comprehensive documentation and inline comments
    - Handle missing data and outliers appropriately
'''

In [ ]:
%%time

for i in range(5):
    task_intention_temp = task_intention + f'''
Output Requirements
    - Save tabular results to: `{base_dir}/result/{study_id}/aging_analysis_prompt1_run{i}.csv`
    - Save analysis report to: `{base_dir}/result/{study_id}/aging_analysis_prompt1_run{i}.md`
    - DO NOT create visualizations or any other things that cost additional tokens'''
    out = agent.go(task_intention_temp)

In [ ]:
ls /dfs/user/hanchenw/Agent_PG/clockbase_agent/result/GSE122080

In [ ]:
RES_DIR = "/dfs/user/hanchenw/Agent_PG/clockbase_agent/result"

pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_run1.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_run2.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_run3.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_run4.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_run0.csv")

In [ ]:
%%time

for i in range(5):

    agent = codeact_agent(coding_tool, llm_name)
    agent.configure()

    task_intention_temp = task_intention + f'''
Output Requirements
    - Save tabular results to: `{base_dir}/result/{study_id}/aging_analysis_prompt1_reinit_run{i}.csv`
    - Save analysis report to: `{base_dir}/result/{study_id}/aging_analysis_prompt1_reinit_run{i}.md`
    - DO NOT create visualizations or any other things that cost additional tokens'''
    out = agent.go(task_intention_temp)

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_reinit_run0.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_reinit_run1.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_reinit_run2.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_reinit_run3.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt1_reinit_run4.csv")

In [ ]:
os.listdir(RES_DIR)

In [ ]:
os.listdir(f"{RES_DIR}/GSE79409")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE237968/aging_analysis.csv")

In [ ]:
from IPython.display import Markdown, display

# Read the markdown file
with open(f"{RES_DIR}/GSE122080/aging_analysis.md", 'r', encoding='utf-8') as f:
    content = f.read()

# Display the markdown content with proper rendering
display(Markdown(content))

### Prompt 2 (CSV Only)
    $0.4 - 0.5 each case

In [ ]:
study_id = "GSE122080"
study_design = get_geo_metadata(study_id)

base_dir = "/dfs/user/hanchenw/Agent_PG/clockbase_agent/"
data_dir = f"{base_dir}/data"

task_intention = f'''
Analyze the aging effects of conditions from study {study_id}.

Workflow:

1. Data Acquisition and Preparation
    - Load dataset overview from: `{data_dir}/dataset_level/gse_rna_mm.csv` (Filter by "accession" column to locate {study_id})
    - Extract metadata from: `{data_dir}/GEO_mm_rna/processed_metadata_preharmonize/{study_id}.csv`
    - Import pre-calculated aging clock data from: `{data_dir}/clock_output/mouse_rna_age_short.csv`
    - Create output directory if needed: `/dfs/user/hanchenw/Agent_PG/clockbase_agent/result/{study_id}/`


2. Study Design Information:
   {study_design}

3. Clock Selection and Analysis:
    - **IMPORTANT: DO NOT compare young and old populations directly**
    - **Target Aging Clocks:**
      * `Hazard_Rodents_All_All_scaleddiff_EN_tAge`
      * `Hazard_Mouse_All_All_scaleddiff_EN_tAge`
      
    - **For Each Condition:**
      * Identify appropriate control group(s) based on metadata
      * Determine relevant covariates (age, sex, tissue type, batch effects)
      * Focus on comparing treatment vs. control within the same age group
      * Perform statistical analysis:
        - Use appropriate tests based on data structure (t-test, ANOVA, or linear models)
        - Report raw p-values (no adjustment required)
        - Calculate standardized effect sizes (Cohen's d preferred)

4. Deliverables:
   - Standardized results table with these columns:
     * gse_id: The dataset identifier ({study_id})
     * comparison_name: Descriptive name of the comparison (e.g., "Drug X vs Vehicle")
     * condition_name: Concise intervention name (e.g., "Rapamycin")
     * condition_category: Categorization (one of: Genetic, Drug, Diet, Environment, Disease, Other)
     * covariate: Variables controlled for in analysis
     * clock: Specific aging clock used
     * statistical_test: Method used (e.g., "ANOVA", "Linear Mixed Model")
     * p_value: Raw p-value (do NOT use adjusted p-values)
     * effect_size: Magnitude of effect (Cohen's d or similar)
     * direction: Age effect direction (1=accelerated aging, -1=decelerated aging)
     * trust_score: Evaluate experimental follow-up value (0-10):
     
         Trust Score Criteria
            - **0-3: Low value**
              - Likely technical artifact
              - Limited biological plausibility
              - Insufficient replication
            
            - **4-6: Moderate value**
              - Some biological plausibility
              - Limited supporting literature
              - Adequate experimental design
            
            - **7-8: High value**
              - Strong biological plausibility
              - Supported by published mechanisms
              - Robust experimental design
            
            - **9-10: Exceptional value**
              - Novel with transformative potential
              - Strong literature support for mechanism
              - Exceptional rigor in design and analysis

Implementation Requirements:
    - Use Python with pandas, scipy, and statsmodels
    - Implement modular code structure with separate functions for:
      * Data loading and preprocessing
      * Statistical analysis
      * Results formatting
    - Include efficient documentation and inline comments
    - Handle missing data and outliers appropriately

Output Requirements
    - Save to: {base_dir}/result/{study_id}/aging_analysis.csv
    - DO NOT generate reports, visualizations, or any other output.
    - DO NOT provide explanations or summaries in the response.
    - Return ONLY a short confirmation message when complete.
'''

In [ ]:
%%time
out = agent.go(task_intention)

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis.csv")

In [ ]:
%%time

for i in range(5):
    task_intention_temp = task_intention + f'''
Output Requirements
    - Save tabular results to: `{base_dir}/result/{study_id}/aging_analysis_prompt2_run{i}.csv`
    - Save analysis report to: `{base_dir}/result/{study_id}/aging_analysis_prompt2_run{i}.md`
    - DO NOT create visualizations or any other things that cost additional tokens'''
    out = agent.go(task_intention_temp)

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt2_run0.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt2_run1.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt2_run2.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt2_run3.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis_prompt2_run4.csv")

### Prompt 3 - Moving Score ahead

In [ ]:
study_id = "GSE208050" # "GSE104282" # "GSE122080"
study_design = get_geo_metadata(study_id)

base_dir = "/dfs/user/hanchenw/Agent_PG/clockbase_agent/"
data_dir = f"{base_dir}/data"

task_intention = f'''
Analyze the aging effects of conditions from study {study_id}.

Workflow:

1. Data Acquisition and Preparation
    * Load dataset overview from: `{data_dir}/dataset_level/gse_rna_mm.csv` (Filter by "accession" column to locate {study_id})
    * Extract metadata from: `{data_dir}/GEO_mm_rna/processed_metadata_preharmonize/{study_id}.csv`
    * Import pre-calculated aging clock data from: `{data_dir}/clock_output/mouse_rna_age_short.csv`
    * Create output directory if needed: `/dfs/user/hanchenw/Agent_PG/clockbase_agent/result_prompt3/{study_id}/`

2. Study Design Information:
   {study_design}

3. Clock Selection and Analysis:
    * **IMPORTANT: DO NOT compare young and old populations directly**
    * **Target Aging Clocks:**
    * `Hazard_Rodents_All_All_scaleddiff_EN_tAge`
    * `Hazard_Mouse_All_All_scaleddiff_EN_tAge`

    * **For Each Condition:**
    * Identify appropriate control group(s) based on metadata
    * Determine relevant covariates (age, sex, tissue type, batch effects)
    * Focus on comparing treatment vs. control within the same age group
    * Before statistical testing, determine the experiment's trust score (0-10):
        Trust score evaluates the experimental validity and translational potential of an aging study based on intervention/perturbation type and biological plausibility, completely independent of statistical outcomes like p-values and effect sizes.

        * **0-3: Low value**
        * Typically immortalized/cancer cell lines 
        * Studies with limited connection to aging biology
        * Minimal literature precedent for mechanism

        * **4-6: Moderate value**
        * Well-designed in vitro studies or simple model organisms
        * Reasonable connection to known aging pathways
        * Some supporting evidence in aging research

        * **7-8: High value**
        * In vivo studies or promising in vitro work with strong mechanistic basis
        * Clear connection to established aging biology
        * Substantial literature support
        * Good translational potential

        * **9-10: Exceptional value**
        * Novel interventions with compelling aging-related mechanisms
        * Robust evidence across experimental systems
        * Exceptional translational potential for human applications
    * Perform statistical analysis:
        * Use appropriate tests based on data structure (t-test, ANOVA, or linear models)
        * Report raw p-values (no adjustment required)
        * Calculate standardized effect sizes

4. Deliverables:
* Standardized results table with these columns:
    * gse_id: The dataset identifier ({study_id})
    * comparison_name: Descriptive name of the comparison (e.g., "Drug X vs Vehicle")
    * condition_name: Concise intervention name (e.g., "Rapamycin")
    * condition_category: Categorization (one of: Genetic, Drug, Diet, Environment, Disease, Other)
    * covariate: Variables controlled for in analysis
    * clock: Specific aging clock used
    * statistical_test: Method used (e.g., "ANOVA", "Linear Mixed Model")
    * p_value: Raw p-value (do NOT use adjusted p-values)
    * effect_size: Magnitude of effect (Cohen's d or similar)
    * direction: Age effect direction (1=accelerated aging, -1=decelerated aging)
    * trust_score: Evaluate experimental follow-up value (0-10):


Implementation Requirements:
    * Use Python with pandas, scipy, and statsmodels
    * Implement modular code structure with separate functions for:
    * Data loading and preprocessing
    * Statistical analysis
    * Results formatting
    * Include efficient documentation and inline comments
    * Handle missing data and outliers appropriately

Output Requirements
    * Save to: {base_dir}/result_prompt3/{study_id}/aging_analysis.csv
    * DO NOT generate reports, visualizations, or any other output.
    * DO NOT provide explanations or summaries in the response.
    * Return ONLY a short confirmation message when complete.
'''

In [ ]:
out = agent.go(task_intention)

In [ ]:
%%time

for i in range(5):
    task_intention_temp = task_intention + f'''
Output Requirements
    - Save tabular results to: `{base_dir}/result/{study_id}/aging_analysis_prompt3_run{i}.csv`
    - Save analysis report to: `{base_dir}/result/{study_id}/aging_analysis_prompt3_run{i}.md`
    - DO NOT create visualizations or any other things that cost additional tokens'''
    out = agent.go(task_intention_temp)

In [ ]:
pd.read_csv(f"{RES_DIR}/{study_id}/aging_analysis_prompt3_run0.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/{study_id}/aging_analysis_prompt3_run1.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/{study_id}/aging_analysis_prompt3_run2.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/{study_id}/aging_analysis_prompt3_run3.csv")

In [ ]:
pd.read_csv(f"{RES_DIR}/{study_id}/aging_analysis_prompt3_run4.csv")

In [ ]:
%%time
out = agent.go(task_intention)

In [ ]:
pd.read_csv(f"{RES_DIR}/GSE122080/aging_analysis.csv")

    The "Wall time" in your output indicates that 2 minutes and 40 seconds passed in real-world time during the execution of whatever code was being run.
    The CPU times show:
    
    User time (time spent executing user code): 318 milliseconds
    System time (time spent in system calls): 624 milliseconds
    Total CPU time: 942 milliseconds (less than 1 second)
    
    The large difference between CPU time (942ms) and Wall time (2min 40s) suggests that most of the time was spent waiting for something outside of CPU processing - likely network requests, I/O operations, or waiting for a remote API response (which would make sense in the context of our earlier discussion about the Anthropic API).

In [ ]:
import anthropic

In [ ]:
    client = anthropic.Anthropic(
        api_key=os.environ.get("ANTHROPIC_API_KEY")
    )
    
    response = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=1,
        system="Count tokens only, no response needed.",
        messages=[{"role": "user", "content": task_intention}]
    )
    
    # Get the token count from the response
    token_count = response.usage.input_tokens
    print(f"Approximate prompt cost: ${token_count * 0.000008:.6f}")

In [ ]:
    client = anthropic.Anthropic(
        api_key=os.environ.get("ANTHROPIC_API_KEY")
    )
    
    response = client.messages.create(
        model="claude-3-7-sonnet-20250219",
        max_tokens=1,
        system="Count tokens only, no response needed.",
        messages=[{"role": "user", "content": agent.system_prompt}]
    )
    
    # Get the token count from the response
    token_count = response.usage.input_tokens
    print(f"Approximate prompt cost: ${token_count * 0.000008:.6f}")

In [ ]:
agent.system_prompt

In [ ]:
20k * 0.1 -> 2k 
40k * 0.1 -> 

In [ ]:
        df = pd.read_csv("/dfs/user/hanchenw/Agent_PG/clockbase_agent/data/clock_output/mouse_rna_age.csv")
        
        # Get unique GSE IDs
        unique_gse_ids = df["gse_id"].unique()
        
        # Convert GSE IDs to numeric values for sorting
        gse_numeric = pd.DataFrame({
            'gse_id': unique_gse_ids,
            'numeric_val': [int(gse_id.replace('GSE', '')) for gse_id in unique_gse_ids]
        })
        
        # Sort by numeric values in descending order and get top 100
        sorted_gse = gse_numeric.sort_values('numeric_val', ascending=False)
        top_100_gse = sorted_gse['gse_id'].head(100).tolist()
        
        # Add the specific GSE IDs to ensure they're included
        study_ids = top_100_gse + ["GSE122080", "GSE139389", "GSE104282"]
        # Remove duplicates while preserving order
        study_ids = list(dict.fromkeys(study_ids))

In [ ]:
study_ids

In [ ]:
gse_numeric